# PTM-Prediction — Fases 1-3b en Colab (GPU)

Ejecuta las fases 1 → 1.5 → 2 → 3 → 3b del pipeline sobre una GPU T4 gratuita, en vez de
CPU local.

**Corre:** saneamiento, extracción de estructura, DeepMVP, DeepPTMPred, fase 3b (vía
secretora, Kinase Library, MeToken, EMNGly, competencia entre PTMs).

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU**.

El disco de la sesión gratuita ronda 112GB y se llena bastante entre entornos conda y
pesos -- cada celda de entorno limpia su caché al terminar (`mamba clean -afy`), pero si
el panel de "Resources" muestra poco espacio libre, ejecuta esa misma orden manualmente
antes de seguir.

Las Secciones 5-9 cachean sus entornos conda ya construidos en Drive con `conda-pack`
(mismo patrón que la Sección 4 con los pesos vía `fetch_or_copy`): la primera sesión
construye y sube cada entorno, las siguientes lo restauran directamente (segundos, en vez
de los ~10-20 min que domina la construcción desde cero). Detalle y por qué no se
paralelizó la construcción en su lugar:
`01-Proyectos/PTM-Prediction/Decisiones/2026-08-09-notebook-colab-fases-1-3b-gpu.md` del
vault.


In [1]:
!nvidia-smi


Tue Aug 11 10:40:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Drive (cache de pesos)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_ROOT = "/content/drive/MyDrive/PTM-Prediction-Colab"
for sub in ["weights/DeepMVP/models",
            "weights/DeepPTMPred/esm",
            "weights/MeToken",
            "weights/EMNgly/esm",
            "weights/EMNgly/checkpoints",
            "envs",
            "outputs_backup"]:
    os.makedirs(f"{CACHE_ROOT}/{sub}", exist_ok=True)
print("Cache en:", CACHE_ROOT)


Mounted at /content/drive
Cache en: /content/drive/MyDrive/PTM-Prediction-Colab


## 2. `condacolab` (Miniforge + `mamba`)

Esta celda reinicia el runtime automáticamente. Es normal — continúa con la celda
siguiente cuando termine.


In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()


✨🍰✨ Everything looks OK!


### Continúa aquí tras el reinicio

In [2]:
import os
# En runtime GPU, Colab fija LD_LIBRARY_PATH=/usr/lib64-nvidia al reiniciar y pisa el
# parche de condacolab -- lo prepende de nuevo antes de check().
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

import condacolab
condacolab.check()
!mamba --version

# Empaqueta/restaura los 5 entornos de las Secciones 5-9 contra Drive -- ver
# Seccion 5 en adelante. Se instala aqui, antes de crear ningun entorno, para
# que este disponible en cuanto la primera seccion lo necesite.
!pip install -q conda-pack

CACHE_ROOT = "/content/drive/MyDrive/PTM-Prediction-Colab"
REPO = "/content/PTM-Prediction"
assert os.path.isdir(CACHE_ROOT), "Drive no está montado -- repite la celda de montaje."


✨🍰✨ Everything looks OK!
2.4.0


## 3. Clonar el pipeline y los 4 repos de motores

In [3]:
%cd /content
!git clone -q https://github.com/Lvera-code/PTM-Prediction.git

%cd {REPO}
!pip install -q -r requirements.txt

# Clonar los 4 repos antes de crear ninguna subcarpeta de checkpoints: git no clona
# sobre un directorio ya existente y no vacío.
!git clone -q https://github.com/bzhanglab/DeepMVP DeepMVP
!git clone -q https://github.com/kuikui-wang/DeepPTMPred DeepPTMPred
!git clone -q https://github.com/A4Bio/MeToken MeToken
!git clone -q https://github.com/StellaHxy/EMNgly EMNgly
print("5 repos clonados.")


/content
/content/PTM-Prediction
5 repos clonados.


## 4. Descargas pesadas en segundo plano

ESM-1b/ESM-2 se descargan con `aria2c` mientras se crean los entornos conda. Si ya están
en Drive de una sesión anterior, se copian en lugar de descargarse. La Sección 12 espera
a que terminen antes de ejecutar el pipeline.


In [4]:
!apt-get -qq install -y aria2 > /dev/null
import os
os.makedirs("/content/dl_logs", exist_ok=True)
DL_LOGS = [
    "/content/dl_logs/esm2_main.log",
    "/content/dl_logs/esm2_reg.log",
    "/content/dl_logs/esm1b_main.log",
    "/content/dl_logs/esm1b_reg.log",
    "/content/dl_logs/nglyde_svm.log",
    "/content/dl_logs/metoken_zip.log",
]
for f in DL_LOGS:
    open(f, "w").close()


In [5]:
%%bash -s "$CACHE_ROOT" "$REPO"
CACHE_ROOT="$1"
REPO="$2"

fetch_or_copy () {
  local tag="$1" drive_path="$2" url="$3" dest="$4" log="$5"
  mkdir -p "$(dirname "$dest")"
  if [ -s "$drive_path" ]; then
    echo "[$tag] copiando desde Drive..." > "$log"
    cp "$drive_path" "$dest"
  else
    echo "[$tag] descargando..." > "$log"
    aria2c -x 16 -s 16 -k 1M -q -o "$(basename "$dest")" -d "$(dirname "$dest")" "$url" >> "$log" 2>&1
    mkdir -p "$(dirname "$drive_path")"
    cp "$dest" "$drive_path"
  fi
  echo "[$tag] listo." >> "$log"
}

fetch_or_copy "esm2-main" \
  "$CACHE_ROOT/weights/DeepPTMPred/esm/esm2_t33_650M_UR50D.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" \
  "$REPO/DeepPTMPred/esm/checkpoints/esm2_t33_650M_UR50D.pt" \
  "/content/dl_logs/esm2_main.log" &

fetch_or_copy "esm2-reg" \
  "$CACHE_ROOT/weights/DeepPTMPred/esm/esm2_t33_650M_UR50D-contact-regression.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" \
  "$REPO/DeepPTMPred/esm/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt" \
  "/content/dl_logs/esm2_reg.log" &

fetch_or_copy "esm1b-main" \
  "$CACHE_ROOT/weights/EMNgly/esm/esm1b_t33_650M_UR50S.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/models/esm1b_t33_650M_UR50S.pt" \
  "$REPO/EMNgly/esm/checkpoints/esm1b_t33_650M_UR50S.pt" \
  "/content/dl_logs/esm1b_main.log" &

fetch_or_copy "esm1b-reg" \
  "$CACHE_ROOT/weights/EMNgly/esm/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/regression/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "$REPO/EMNgly/esm/checkpoints/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "/content/dl_logs/esm1b_reg.log" &

fetch_or_copy "nglyde-svm" \
  "$CACHE_ROOT/weights/EMNgly/checkpoints/N-GlyDE.pickle" \
  "https://drive.usercontent.google.com/download?id=1hbnEtHHXTGnQAFm-cCHMj3pWQiAYAUsw&export=download&confirm=t" \
  "$REPO/EMNgly/checkpoints/N-GlyDE.pickle" \
  "/content/dl_logs/nglyde_svm.log" &

(
  if [ -s "$CACHE_ROOT/weights/MeToken/pretrained_model.zip" ]; then
    echo "[metoken-zip] copiando desde Drive..." > /content/dl_logs/metoken_zip.log
    cp "$CACHE_ROOT/weights/MeToken/pretrained_model.zip" /tmp/pretrained_model.zip
  else
    echo "[metoken-zip] descargando..." > /content/dl_logs/metoken_zip.log
    curl -sL -o /tmp/pretrained_model.zip \
      https://github.com/A4Bio/MeToken/releases/download/1.0/pretrained_model.zip
    cp /tmp/pretrained_model.zip "$CACHE_ROOT/weights/MeToken/pretrained_model.zip"
  fi
  echo "[metoken-zip] listo." >> /content/dl_logs/metoken_zip.log
) &

disown -a
echo "6 descargas arrancadas en segundo plano."
echo "(Progreso: !tail -n2 /content/dl_logs/*.log)"


6 descargas arrancadas en segundo plano.
(Progreso: !tail -n2 /content/dl_logs/*.log)


## 5. DeepMVP

Motor obligatorio (no opcional): sin sus pesos, `pipeline.py` falla al ejecutar. No
tienen URL directa (web con clic manual en `deepmvp.ptmax.org`).

Comprime la carpeta local `DeepMVP/models/` en un único zip (subir una carpeta entera por
el navegador pierde archivos -- confirmado dos veces) y sube ESE archivo, no la carpeta:

```bash
cd DeepMVP/models
zip -r ../models.zip .
```

Sube `models.zip` a `Drive/PTM-Prediction-Colab/weights/DeepMVP/models.zip` -- las
siguientes sesiones ya lo encuentran ahí.


In [6]:
%%bash -s "$CACHE_ROOT" "$REPO"
set -e
CACHE_ROOT="$1"
REPO="$2"
ENV_NAME="deepmvp"
ENV_TAR="$CACHE_ROOT/envs/${ENV_NAME}.tar.gz"

if [ -s "$ENV_TAR" ]; then
  echo "[$ENV_NAME] restaurando entorno cacheado desde Drive..."
  mkdir -p "/usr/local/envs/$ENV_NAME"
  tar -xzf "$ENV_TAR" -C "/usr/local/envs/$ENV_NAME"
  "/usr/local/envs/$ENV_NAME/bin/conda-unpack"
  echo "[$ENV_NAME] restaurado."
else
  echo "[$ENV_NAME] sin cache en Drive -- construyendo desde cero..."
  cd "$REPO"
  # pyteomics=4.4.2 no existe en conda-forge (solo en PyPI) -- se instala por
  # separado en vez de usar environment.yml directo, que falla al resolver
  # por ese paquete.
  mamba create -q -n "$ENV_NAME" python=3.7.10 -y
  mamba install -q -n "$ENV_NAME" -c conda-forge -y ipython numpy=1.19.5 h5py=2.10.0 pandas=1.2.4 scikit-learn=0.24.2 matplotlib=3.4.2 biopython=1.78 shap=0.39.0 cudatoolkit=11.0 "cudnn=8.0.*"
  mamba run -n "$ENV_NAME" pip install -q pyteomics==4.4.2 tensorflow==2.4.2
  mamba clean -afy -q

  echo "[$ENV_NAME] empaquetando y subiendo a Drive para la proxima sesion..."
  conda-pack -n "$ENV_NAME" -o "/tmp/${ENV_NAME}.tar.gz" -q --ignore-missing-files
  cp "/tmp/${ENV_NAME}.tar.gz" "$ENV_TAR"
  rm -f "/tmp/${ENV_NAME}.tar.gz"
  echo "[$ENV_NAME] cacheado."
fi


[deepmvp] restaurando entorno cacheado desde Drive...
[deepmvp] restaurado.


In [7]:
DEEPMVP_PYTHON_BIN = !mamba run -n deepmvp which python
DEEPMVP_PYTHON_BIN = DEEPMVP_PYTHON_BIN[0]
print("DEEPMVP_PYTHON_BIN =", DEEPMVP_PYTHON_BIN)


DEEPMVP_PYTHON_BIN = /usr/local/envs/deepmvp/bin/python


In [8]:
import os, shutil, zipfile
drive_zip = f"{CACHE_ROOT}/weights/DeepMVP/models.zip"
local_models = f"{REPO}/DeepMVP/models"

if os.path.isfile(drive_zip):
    shutil.rmtree(local_models, ignore_errors=True)
    with zipfile.ZipFile(drive_zip) as zf:
        zf.extractall(local_models)
    print("Pesos extraídos:", os.listdir(local_models))
else:
    print(f"No hay 'models.zip' en Drive todavía. Súbelo a: {drive_zip}")
    print("DeepMVP es obligatorio -- sin pesos, pipeline.py falla al ejecutar (Sección 15).")
    print("Vuelve a correr esta celda cuando esté subido.")


Pesos extraídos: ['methylation_k', 'ubiquitination_k', 'methylation_r', 'sumoylation_k', 'phosphorylation_st', 'phosphorylation_y', 'glycosylation_n', 'acetylation_k']


## 6. DeepPTMPred

In [9]:
%%bash -s "$CACHE_ROOT" "$REPO"
set -e
CACHE_ROOT="$1"
REPO="$2"
ENV_NAME="deepptmpred"
ENV_TAR="$CACHE_ROOT/envs/${ENV_NAME}.tar.gz"

if [ -s "$ENV_TAR" ]; then
  echo "[$ENV_NAME] restaurando entorno cacheado desde Drive..."
  mkdir -p "/usr/local/envs/$ENV_NAME"
  tar -xzf "$ENV_TAR" -C "/usr/local/envs/$ENV_NAME"
  "/usr/local/envs/$ENV_NAME/bin/conda-unpack"
  echo "[$ENV_NAME] restaurado."
else
  echo "[$ENV_NAME] sin cache en Drive -- construyendo desde cero..."
  cd "$REPO"
  # El environment.yml de DeepPTMPred (upstream, no editable desde aqui) fija
  # "tensorflow=2.15" en la seccion pip: con sintaxis de conda -- pip exige "==",
  # si no rechaza el requirement y mata la creacion del entorno.
  sed -i 's/tensorflow=2\.15/tensorflow==2.15/' DeepPTMPred/pred/train_PTM/environment.yml
  mamba env create -q -f DeepPTMPred/pred/train_PTM/environment.yml -n "$ENV_NAME"
  # El environment.yml no fija version de mkl -- mamba resuelve una demasiado nueva,
  # incompatible con el build de pytorch=2.0 (ImportError: undefined symbol
  # iJIT_NotifyEvent al importar torch). mkl==2021.4.0 es la version conocida-buena.
  mamba install -q -n "$ENV_NAME" -c conda-forge "mkl=2021.4.0" -y
  # predict.py importa pyrosetta a nivel de modulo y SI lo usa en tiempo de
  # ejecucion (PyRosettaCalculator, SASA por residuo) -- no esta en el
  # environment.yml (no distribuible via pip/conda normal). Mismo metodo que
  # documenta STATUS.md para Fase A: pyrosetta-installer, licencia academica
  # automatica.
  mamba run -n "$ENV_NAME" pip install -q pyrosetta-installer

  # install_pyrosetta() por defecto usa el mirror West (rosettacommons.org), que
  # para el wheel python3.10/ubuntu sirve un placeholder roto
  # ("pyrosetta-0-...whl", 404 al descargar). El mirror East (JHU graylab)
  # resuelve la version real, pero ese servidor no manda el certificado
  # intermedio de su cadena TLS -- falla la verificacion incluso via curl con el
  # bundle completo del sistema, fuera de Python. install_pyrosetta() no permite
  # pasarle --trusted-host al pip interno, asi que se replica el mismo flujo a
  # mano: TLS sin verificar solo contra ese host academico (no viajan
  # credenciales -- el propio paquete usa login/password vacios).
  cat > /tmp/install_pyrosetta.py << 'PYEOF'
import ssl, urllib.request, subprocess

ctx = ssl._create_unverified_context()
url_dir = "https://graylab.jhu.edu/download/PyRosetta4/archive/release/PyRosetta4.Release.python310.ubuntu.wheel/"
with urllib.request.urlopen(url_dir + "latest.html", context=ctx) as f:
    html = f.read().decode("utf-8")
wheel = html.partition("url=")[2].partition('"')[0]
wheel_url = url_dir + wheel
print("PyRosetta wheel:", wheel_url)
subprocess.check_call(["pip", "install", "--trusted-host", "graylab.jhu.edu", wheel_url])
subprocess.check_call(["pip", "install", "-q", "numpy"])
PYEOF
  mamba run -n "$ENV_NAME" python /tmp/install_pyrosetta.py
  mamba clean -afy -q

  echo "[$ENV_NAME] empaquetando y subiendo a Drive para la proxima sesion..."
  conda-pack -n "$ENV_NAME" -o "/tmp/${ENV_NAME}.tar.gz" -q --ignore-missing-files
  cp "/tmp/${ENV_NAME}.tar.gz" "$ENV_TAR"
  rm -f "/tmp/${ENV_NAME}.tar.gz"
  echo "[$ENV_NAME] cacheado."
fi


[deepptmpred] restaurando entorno cacheado desde Drive...
[deepptmpred] restaurado.


In [10]:
DEEPPTMPRED_PYTHON_BIN = !mamba run -n deepptmpred which python
DEEPPTMPRED_PYTHON_BIN = DEEPPTMPRED_PYTHON_BIN[0]
print("DEEPPTMPRED_PYTHON_BIN =", DEEPPTMPRED_PYTHON_BIN)


DEEPPTMPRED_PYTHON_BIN = /usr/local/envs/deepptmpred/bin/python


## 7. MeToken

In [11]:
%%bash -s "$CACHE_ROOT" "$REPO"
set -e
CACHE_ROOT="$1"
REPO="$2"
ENV_NAME="metoken"
ENV_TAR="$CACHE_ROOT/envs/${ENV_NAME}.tar.gz"

if [ -s "$ENV_TAR" ]; then
  echo "[$ENV_NAME] restaurando entorno cacheado desde Drive..."
  mkdir -p "/usr/local/envs/$ENV_NAME"
  tar -xzf "$ENV_TAR" -C "/usr/local/envs/$ENV_NAME"
  "/usr/local/envs/$ENV_NAME/bin/conda-unpack"
  echo "[$ENV_NAME] restaurado."
else
  echo "[$ENV_NAME] sin cache en Drive -- construyendo desde cero..."
  cd "$REPO"
  mamba create -q -n "$ENV_NAME" python=3.10 -y
  mamba run -n "$ENV_NAME" pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
  # transformers sin pinear instala la ultima version -- su codigo interno de
  # sharding (hub_kernels/core_model_loading, muy reciente) importa
  # torch.distributed.tensor.DTensor, API que no existe igual en torch==2.4.0
  # (ImportError). 4.44.2 es contemporanea de torch 2.4 (ambas ~jul-ago 2024),
  # sin ese codigo de sharding todavia.
  mamba run -n "$ENV_NAME" pip install -q "transformers==4.44.2" numpy scipy biopython omegaconf tqdm pandas huggingface-hub h5py

  mamba run -n "$ENV_NAME" pip install -q torch_scatter -f https://data.pyg.org/whl/torch-2.4.0+cu121.html \
    || (echo "Sin wheel prebuilt -- compilando desde fuente." && mamba run -n "$ENV_NAME" pip install -q torch_scatter)
  mamba clean -afy -q
  mamba run -n "$ENV_NAME" pip cache purge -q

  echo "[$ENV_NAME] empaquetando y subiendo a Drive para la proxima sesion..."
  conda-pack -n "$ENV_NAME" -o "/tmp/${ENV_NAME}.tar.gz" -q --ignore-missing-files
  cp "/tmp/${ENV_NAME}.tar.gz" "$ENV_TAR"
  rm -f "/tmp/${ENV_NAME}.tar.gz"
  echo "[$ENV_NAME] cacheado."
fi


[metoken] restaurando entorno cacheado desde Drive...
[metoken] restaurado.


In [12]:
METOKEN_PYTHON_BIN = !mamba run -n metoken which python
METOKEN_PYTHON_BIN = METOKEN_PYTHON_BIN[0]
print("METOKEN_PYTHON_BIN =", METOKEN_PYTHON_BIN)


METOKEN_PYTHON_BIN = /usr/local/envs/metoken/bin/python


## 8. EMNGly

Los pesos del SVM se entrenaron con `scikit-learn==1.1.1` -- no actualizar esa versión.


In [13]:
%%bash -s "$CACHE_ROOT" "$REPO"
set -e
CACHE_ROOT="$1"
REPO="$2"
ENV_NAME="emngly"
ENV_TAR="$CACHE_ROOT/envs/${ENV_NAME}.tar.gz"

if [ -s "$ENV_TAR" ]; then
  echo "[$ENV_NAME] restaurando entorno cacheado desde Drive..."
  mkdir -p "/usr/local/envs/$ENV_NAME"
  tar -xzf "$ENV_TAR" -C "/usr/local/envs/$ENV_NAME"
  "/usr/local/envs/$ENV_NAME/bin/conda-unpack"
  echo "[$ENV_NAME] restaurado."
else
  echo "[$ENV_NAME] sin cache en Drive -- construyendo desde cero..."
  cd "$REPO"
  # scikit-learn==1.1.1 no tiene wheel para Python 3.12 (el python3 base de Colab) --
  # entorno propio con python=3.10, que sí la tiene.
  mamba create -q -n "$ENV_NAME" python=3.10 -y
  mamba run -n "$ENV_NAME" pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
  mamba run -n "$ENV_NAME" pip install -q fair-esm "scikit-learn==1.1.1" scipy pandas tqdm wget
  mamba run -n "$ENV_NAME" pip install -q "numpy==1.23.5"
  mamba clean -afy -q
  mamba run -n "$ENV_NAME" pip cache purge -q

  echo "[$ENV_NAME] empaquetando y subiendo a Drive para la proxima sesion..."
  conda-pack -n "$ENV_NAME" -o "/tmp/${ENV_NAME}.tar.gz" -q --ignore-missing-files
  cp "/tmp/${ENV_NAME}.tar.gz" "$ENV_TAR"
  rm -f "/tmp/${ENV_NAME}.tar.gz"
  echo "[$ENV_NAME] cacheado."
fi


[emngly] restaurando entorno cacheado desde Drive...
[emngly] restaurado.


In [14]:
EMNGLY_PYTHON_BIN = !mamba run -n emngly which python
EMNGLY_PYTHON_BIN = EMNGLY_PYTHON_BIN[0]
print("EMNGLY_PYTHON_BIN =", EMNGLY_PYTHON_BIN)


EMNGLY_PYTHON_BIN = /usr/local/envs/emngly/bin/python


## 9. Kinase Library

In [15]:
%%bash -s "$CACHE_ROOT"
set -e
CACHE_ROOT="$1"
ENV_NAME="kinase_library"
ENV_TAR="$CACHE_ROOT/envs/${ENV_NAME}.tar.gz"

if [ -s "$ENV_TAR" ]; then
  echo "[$ENV_NAME] restaurando entorno cacheado desde Drive..."
  mkdir -p "/usr/local/envs/$ENV_NAME"
  tar -xzf "$ENV_TAR" -C "/usr/local/envs/$ENV_NAME"
  "/usr/local/envs/$ENV_NAME/bin/conda-unpack"
  echo "[$ENV_NAME] restaurado."
else
  echo "[$ENV_NAME] sin cache en Drive -- construyendo desde cero..."
  mamba create -q -n "$ENV_NAME" python=3.10 -y
  mamba run -n "$ENV_NAME" pip install -q kinase-library
  mamba clean -afy -q

  echo "[$ENV_NAME] empaquetando y subiendo a Drive para la proxima sesion..."
  conda-pack -n "$ENV_NAME" -o "/tmp/${ENV_NAME}.tar.gz" -q --ignore-missing-files
  cp "/tmp/${ENV_NAME}.tar.gz" "$ENV_TAR"
  rm -f "/tmp/${ENV_NAME}.tar.gz"
  echo "[$ENV_NAME] cacheado."
fi


[kinase_library] restaurando entorno cacheado desde Drive...
[kinase_library] restaurado.


In [16]:
KINASE_LIBRARY_PYTHON_BIN = !mamba run -n kinase_library which python
KINASE_LIBRARY_PYTHON_BIN = KINASE_LIBRARY_PYTHON_BIN[0]
print("KINASE_LIBRARY_PYTHON_BIN =", KINASE_LIBRARY_PYTHON_BIN)


KINASE_LIBRARY_PYTHON_BIN = /usr/local/envs/kinase_library/bin/python


## 10. Variables de entorno

In [17]:
import os

os.environ["DEEPMVP_PYTHON_BIN"] = DEEPMVP_PYTHON_BIN
os.environ["DEEPMVP_HOME"] = f"{REPO}/DeepMVP"
os.environ["DEEPMVP_MODEL_DIR"] = f"{REPO}/DeepMVP/models"

os.environ["DEEPPTMPRED_PYTHON_BIN"] = DEEPPTMPRED_PYTHON_BIN
os.environ["DEEPPTMPRED_HOME"] = f"{REPO}/DeepPTMPred"

os.environ["METOKEN_PYTHON_BIN"] = METOKEN_PYTHON_BIN
os.environ["METOKEN_HOME"] = f"{REPO}/MeToken"
os.environ["METOKEN_ENABLED"] = "true"

os.environ["EMNGLY_PYTHON_BIN"] = EMNGLY_PYTHON_BIN
os.environ["EMNGLY_HOME"] = f"{REPO}/EMNgly"
os.environ["EMNGLY_ENABLED"] = "true"

os.environ["KINASE_LIBRARY_PYTHON_BIN"] = KINASE_LIBRARY_PYTHON_BIN
os.environ["KINASE_LIBRARY_ENABLED"] = "true"

os.environ["FASTA_INPUT_DIR"] = f"{REPO}/inputs"
os.environ["FASTA_OUTPUT_DIR"] = f"{REPO}/outputs"

print("Variables de entorno listas.")


Variables de entorno listas.


## 11. Esperar las descargas en segundo plano

In [18]:
import time

def wait_for_downloads(logs, poll=15, timeout=3600):
    start = time.time()
    pending = set(logs)
    while pending:
        done_now = {log for log in pending if os.path.exists(log) and "listo." in open(log).read()}
        pending -= done_now
        if not pending:
            break
        if time.time() - start > timeout:
            raise TimeoutError(f"Timeout esperando: {pending}")
        print(f"[{int(time.time() - start)}s] esperando {len(pending)}/{len(logs)} descarga(s)...")
        time.sleep(poll)
    print("Descargas completas.")

wait_for_downloads(DL_LOGS)


Descargas completas.


## 12. Extraer los pesos de MeToken

In [19]:
import zipfile
zipfile.ZipFile('/tmp/pretrained_model.zip').extractall(REPO + '/MeToken')
!ls {REPO}/MeToken/pretrained_model/


checkpoint.ckpt  lightning_checkpoint.ckpt


## 13. Elegir el input a ejecutar

Un input = un archivo FASTA o PDB/mmCIF, con una o varias secuencias dentro (todas se
procesan en la misma ejecución). Elige un candidato del panel o sube el tuyo -- cada
ejecución procesa solo el que esté seleccionado aquí.


In [24]:
import os
from google.colab import files

candidato = "kit_ligand_scf_P21583.pdb"  #@param ["p53_P04637.pdb", "hif1a_Q16665.pdb", "histone_h3_P68431.pdb", "histone_h4_P62805.pdb", "prothrombin_P00734.pdb", "epo_P01588.pdb", "kit_ligand_scf_P21583.pdb", "(subir un archivo nuevo)"]

if candidato == "(subir un archivo nuevo)":
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.rename(fname, f"{REPO}/inputs/{fname}")
    input_file = f"{REPO}/inputs/{fname}"
else:
    input_file = f"{REPO}/inputs/{candidato}"

print("Input seleccionado:", input_file)


Input seleccionado: /content/PTM-Prediction/inputs/kit_ligand_scf_P21583.pdb


## 14. Ejecutar

In [25]:
%cd {REPO}
!python pipeline.py --input {input_file} --output-dir outputs


/content/PTM-Prediction

FASE 1.5 | Extraccion de estructura
Cadena 'A' (273 residuo(s)) -> outputs/kit_ligand_scf_P21583_chain_A.pdb

FASE 2 | Motores (DeepMVP + DeepPTMPred)
DeepMVP: 164 prediccion(es) crudas | DeepPTMPred: 345 prediccion(es) crudas

FASE 3 | Consenso + anotacion
152/366 sitio(s) pasan el umbral | 22 con consenso real (2+ motores de acuerdo)
Por tipo: ubiquitination 19 . o_linked_glycosylation 16 . phosphorylation 16 . gamma_carboxyglutamic_acid 13 . sumoylation 13 . acetylation 13 . crotonylation 12 . malonylation 10 . lys_methylation 9 . n_linked_glycosylation 7 . citrullination 7 . succinylation 4 . arg_methylation 4 . s_nitrosylation 3 . glutarylation 3 . glutathionylation 2 . hydroxylation 1

-- Consenso real (22 sitio(s)) --
Literatura: fuente = panel curado (cada PMID verificado a mano contra NCBI eutils).
Pos  Res  Tipo                    Motor        DMVP   DPTMP  Avisos                                 Quinasa  Literatura  PMIDs  
---  ---  -----------------

In [28]:
%cd {REPO}
!python scripts/validate_biological_panel.py --only p53 kit_ligand_scf --output-dir outputs/validation_panel

/content/PTM-Prediction

=== p53 (P04637, 393 aa) ===
  Tier A: 14/15 sitios reales recuperados (93%)
  Tier B: 18/19 sitios reales recuperados (95%)

=== kit_ligand_scf (P21583, 273 aa) ===
  Tier A: 5/5 sitios reales recuperados (100%)
    control negativo 97 (n_linked_glycosylation): FALSO POSITIVO (el pipeline SI lo acepto)

=== Resumen global ===
Tier A: 19/20 (95%)
Tier B: 18/19 (95%)


## 15. Resultados

In [29]:
import glob, pandas as pd

report = sorted(glob.glob(f"{REPO}/outputs/*_ptm_sites.csv"), key=os.path.getmtime)[-1]
df = pd.read_csv(report)
print(f"{len(df)} sitio(s) en {report}")
df.sort_values("posicion").head(30)


152 sitio(s) en /content/PTM-Prediction/outputs/kit_ligand_scf_P21583_ptm_sites.csv


,accession,posicion,residuo_wt,tipo_ptm,motor,score_deepmvp,score_deepptmpred,consenso,ventana,camino,...,score_emngly,metoken_type,metoken_probability,metoken_type_coincide,via_secretora_evidencia,kinase_library_top_kinase,kinase_library_top_family,kinase_library_percentile,kinase_library_top3_kinases,ptm_crosstalk_aviso
0,kit_ligand_scf_P21583,2,K,lys_methylation,DeepMVP+DeepPTMPred,0.973407,0.369648,False,NaN,PDB,...,NaN,Ubiquitination,0.487376,False,NaN,NaN,NaN,NaN,NaN,"Compite con: crotonylation, glutarylation, mal..."
126,kit_ligand_scf_P21583,2,K,glutarylation,DeepPTMPred,NaN,0.523460,False,NaN,PDB,...,NaN,Ubiquitination,0.487376,NaN,NaN,NaN,NaN,NaN,NaN,"Compite con: crotonylation, lys_methylation, m..."
105,kit_ligand_scf_P21583,2,K,crotonylation,DeepPTMPred,NaN,0.928414,False,NaN,PDB,...,NaN,Ubiquitination,0.487376,NaN,NaN,NaN,NaN,NaN,NaN,"Compite con: glutarylation, lys_methylation, m..."
95,kit_ligand_scf_P21583,2,K,malonylation,DeepPTMPred,NaN,0.652828,False,NaN,PDB,...,NaN,Ubiquitination,0.487376,False,NaN,NaN,NaN,NaN,NaN,"Compite con: crotonylation, glutarylation, lys..."
106,kit_ligand_scf_P21583,3,K,crotonylation,DeepPTMPred,NaN,0.924097,False,NaN,PDB,...,NaN,Ubiquitination,0.606518,NaN,NaN,NaN,NaN,NaN,NaN,"Compite con: glutarylation, malonylation (mism..."
96,kit_ligand_scf_P21583,3,K,malonylation,DeepPTMPred,NaN,0.664760,False,NaN,PDB,...,NaN,Ubiquitination,0.606518,False,NaN,NaN,NaN,NaN,NaN,"Compite con: crotonylation, glutarylation (mis..."
127,kit_ligand_scf_P21583,3,K,glutarylation,DeepPTMPred,NaN,0.513570,False,NaN,PDB,...,NaN,Ubiquitination,0.606518,NaN,NaN,NaN,NaN,NaN,NaN,"Compite con: crotonylation, malonylation (mism..."
136,kit_ligand_scf_P21583,4,T,o_linked_glycosylation,DeepPTMPred,NaN,0.277076,False,NaN,PDB,...,NaN,Phosphorylation,0.846061,False,NaN,NaN,NaN,NaN,NaN,NaN
123,kit_ligand_scf_P21583,11,C,s_nitrosylation,DeepPTMPred,NaN,0.519404,False,NaN,PDB,...,NaN,S-palmitoylation,0.762342,False,NaN,NaN,NaN,NaN,NaN,NaN
107,kit_ligand_scf_P21583,24,K,crotonylation,DeepPTMPred,NaN,0.929667,False,NaN,PDB,...,NaN,Ubiquitination,0.609688,NaN,NaN,NaN,NaN,NaN,NaN,"Compite con: sumoylation, ubiquitination (mism..."


In [30]:
import shutil
shutil.copy(report, f"{CACHE_ROOT}/outputs_backup/{os.path.basename(report)}")
print("Copia en Drive:", f"{CACHE_ROOT}/outputs_backup/{os.path.basename(report)}")


Copia en Drive: /content/drive/MyDrive/PTM-Prediction-Colab/outputs_backup/kit_ligand_scf_P21583_ptm_sites.csv
